# Фінальний проєкт. Статистика інфекційних захворювань у PostgreSQL

## 1. Завантаження даних

### 1.1. Підключення до PostgreSQL

In [1]:
import os
import sys

if "google.colab" in sys.modules:
    os.environ.setdefault("XDG_RUNTIME_DIR", "/tmp")
    os.environ.setdefault("PGSERVER_POSTGRES_VERSION", "16")
    !pip install -q "sqlalchemy>=2.0,<2.1" psycopg2-binary "pandas>=2.2,<3.1"
    !pip install -q pgserver 2>/dev/null || pip install -q pixeltable-pgserver

try:
    import pgserver
except ModuleNotFoundError:
    import pixeltable_pgserver as pgserver

from urllib.request import urlretrieve

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

pg = pgserver.get_server("/tmp/hw_fp_pg", cleanup_mode="stop")
engine = create_engine(pg.get_uri(), future=True)
conn = engine.connect()

print(conn.execute(text("SELECT version()")).scalar_one())
conn.commit()

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
def query(sql, **params):
    df = pd.read_sql(text(sql), conn, params=params)
    conn.commit()
    return df


def execute(sql, **params):
    with conn.begin():
        result = conn.execute(text(sql), params)
        if result.returns_rows:
            return pd.DataFrame(result.fetchall(), columns=list(result.keys()))
    return None


def copy_from_csv(copy_sql, path):
    with conn.begin():
        cursor = conn.connection.dbapi_connection.cursor()
        with open(path, "r", encoding="utf-8", newline="") as file:
            cursor.copy_expert(copy_sql, file)
        cursor.close()

### 1.2. Схема `pandemic`

In [3]:
execute("CREATE SCHEMA IF NOT EXISTS pandemic")
execute("SET search_path TO pandemic")
query("SELECT current_schema() AS current_schema, current_setting('search_path') AS search_path")

,current_schema,search_path
0,pandemic,pandemic


### 1.3. Імпорт `infectious_cases.csv`

In [4]:
CSV_PATH = os.path.join("data", "infectious_cases.csv")
CSV_URL = "https://drive.google.com/uc?export=download&id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"

if not os.path.exists(CSV_PATH):
    os.makedirs("data", exist_ok=True)
    urlretrieve(CSV_URL, CSV_PATH)

print(CSV_PATH, os.path.getsize(CSV_PATH), "bytes")

data/infectious_cases.csv 540895 bytes


In [5]:
execute("""
DROP TABLE IF EXISTS infectious_cases_norm;
DROP TABLE IF EXISTS entities;
DROP TABLE IF EXISTS infectious_cases;

CREATE TABLE infectious_cases (
    "Entity" TEXT,
    "Code" TEXT,
    "Year" TEXT,
    "Number_yaws" TEXT,
    "polio_cases" TEXT,
    "cases_guinea_worm" TEXT,
    "Number_rabies" TEXT,
    "Number_malaria" TEXT,
    "Number_hiv" TEXT,
    "Number_tuberculosis" TEXT,
    "Number_smallpox" TEXT,
    "Number_cholera_cases" TEXT
)
""")

copy_from_csv(
    """
    COPY infectious_cases
    FROM STDIN
    WITH (
        FORMAT csv,
        HEADER true,
        DELIMITER ',',
        FORCE_NOT_NULL (
            "Code", "Number_yaws", "polio_cases", "cases_guinea_worm",
            "Number_rabies", "Number_malaria", "Number_hiv",
            "Number_tuberculosis", "Number_smallpox", "Number_cholera_cases"
        )
    )
    """,
    CSV_PATH,
)

query("SELECT COUNT(*) AS loaded_rows FROM infectious_cases")

,loaded_rows
0,10521


`FORCE_NOT_NULL (...)` залишає порожні поля перелічених колонок порожніми рядками `''`, тобто так, як вони записані у файлі. Без цієї опції `COPY` перетворив би їх на `NULL`.

### 1.4. Огляд даних

In [6]:
query("""
SELECT
    "Entity", "Code", "Year", "Number_yaws", "polio_cases", "cases_guinea_worm",
    "Number_rabies", "Number_malaria", "Number_hiv", "Number_tuberculosis",
    "Number_smallpox", "Number_cholera_cases"
FROM infectious_cases
LIMIT 10
""")

,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,
5,Afghanistan,AFG,1985,,13867,0,,,,,,
6,Afghanistan,AFG,1986,,12901,0,,,,,,
7,Afghanistan,AFG,1987,,4396,0,,,,,,
8,Afghanistan,AFG,1988,,2149,0,,,,,,
9,Afghanistan,AFG,1989,,385,0,,,,,,


In [7]:
query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ("Entity", "Code")) AS entity_code_pairs,
    COUNT(DISTINCT "Entity") AS entities,
    COUNT(DISTINCT "Entity") FILTER (WHERE "Code" = '') AS entities_without_code,
    MIN("Year"::INT) AS min_year,
    MAX("Year"::INT) AS max_year
FROM infectious_cases
""")

,total_rows,entity_code_pairs,entities,entities_without_code,min_year,max_year
0,10521,245,245,34,1920,2023


In [8]:
query("""
SELECT
    column_name,
    COUNT(*) FILTER (WHERE value = '') AS empty_values,
    COUNT(*) FILTER (WHERE value <> '') AS filled_values,
    COUNT(*) FILTER (WHERE value ~ '[.]') AS with_fraction
FROM infectious_cases
CROSS JOIN LATERAL (
    VALUES
        ('Number_yaws', "Number_yaws"),
        ('polio_cases', "polio_cases"),
        ('cases_guinea_worm', "cases_guinea_worm"),
        ('Number_rabies', "Number_rabies"),
        ('Number_malaria', "Number_malaria"),
        ('Number_hiv', "Number_hiv"),
        ('Number_tuberculosis', "Number_tuberculosis"),
        ('Number_smallpox', "Number_smallpox"),
        ('Number_cholera_cases', "Number_cholera_cases")
) AS v(column_name, value)
GROUP BY column_name
ORDER BY empty_values
""")

,column_name,empty_values,filled_values,with_fraction
0,cases_guinea_worm,2093,8428,0
1,polio_cases,2911,7610,0
2,Number_hiv,3741,6780,6729
3,Number_rabies,3741,6780,6780
4,Number_tuberculosis,3741,6780,6649
5,Number_malaria,3741,6780,2672
6,Number_cholera_cases,7554,2967,0
7,Number_smallpox,10417,104,0
8,Number_yaws,10447,74,0


У таблиці 10 521 рядок і 245 пар Entity/Code. Кожна назва трапляється лише з одним кодом, тому пара по суті визначається назвою. Коду немає у 34 сутностей: це континенти, регіони ВООЗ і Світового банку, групи країн за доходом, G20, а також England, Scotland, Wales і Northern Ireland. Пар «країна + рік» з повтором у файлі немає.

Пропусків багато. Number_rabies, Number_malaria, Number_hiv і Number_tuberculosis порожні в одних і тих самих 3 741 рядках. Усі 6 780 заповнених значень Number_rabies дробові, тобто це модельні оцінки, а не підраховані випадки. Number_yaws і Number_smallpox майже порожні: 74 і 104 значення.

## 2. Нормалізація до 3НФ

У сирій таблиці назва і код повторюються в кожному рядку, хоча залежать лише від країни, а не від року. Їх виносимо в довідник `entities`, а в таблиці фактів `infectious_cases_norm` лишається `entity_id` із зовнішнім ключем. Ключ таблиці фактів тепер `(entity_id, year)`, усі показники залежать саме від нього, транзитивних залежностей не лишилось. Це і є 3НФ.

Для регіонів код записується як `NULL`, а не порожній рядок. Звичайний `UNIQUE` вважає два `NULL` різними значеннями, тому в обмеженні стоїть `NULLS NOT DISTINCT` (з'явилось у PostgreSQL 15). З тієї ж причини JOIN зроблено через `IS NOT DISTINCT FROM`. Під час вставки текст отримує типи: цілі лічильники йдуть у `BIGINT`, дробові оцінки в `NUMERIC`.

In [9]:
execute("""
CREATE TABLE entities (
    id SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

CREATE TABLE infectious_cases_norm (
    id SERIAL PRIMARY KEY,
    entity_id INT NOT NULL REFERENCES entities (id),
    year SMALLINT NOT NULL,
    number_yaws BIGINT,
    polio_cases BIGINT,
    cases_guinea_worm BIGINT,
    number_rabies NUMERIC,
    number_malaria NUMERIC,
    number_hiv NUMERIC,
    number_tuberculosis NUMERIC,
    number_smallpox BIGINT,
    number_cholera_cases BIGINT,
    UNIQUE (entity_id, year)
)
""")

In [10]:
execute("""
INSERT INTO entities (entity, code)
SELECT DISTINCT "Entity", NULLIF("Code", '')
FROM infectious_cases
ORDER BY 1
RETURNING id, entity, code
""").head(10)

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,NaN
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


In [11]:
execute("""
INSERT INTO infectious_cases_norm (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    e.id,
    ic."Year"::SMALLINT,
    NULLIF(ic."Number_yaws", '')::BIGINT,
    NULLIF(ic."polio_cases", '')::BIGINT,
    NULLIF(ic."cases_guinea_worm", '')::BIGINT,
    NULLIF(ic."Number_rabies", '')::NUMERIC,
    NULLIF(ic."Number_malaria", '')::NUMERIC,
    NULLIF(ic."Number_hiv", '')::NUMERIC,
    NULLIF(ic."Number_tuberculosis", '')::NUMERIC,
    NULLIF(ic."Number_smallpox", '')::BIGINT,
    NULLIF(ic."Number_cholera_cases", '')::BIGINT
FROM infectious_cases AS ic
JOIN entities AS e
    ON e.entity = ic."Entity"
   AND e.code IS NOT DISTINCT FROM NULLIF(ic."Code", '')
ORDER BY e.id, ic."Year"::SMALLINT
RETURNING id, entity_id, year, number_rabies
""").head(10)

,id,entity_id,year,number_rabies
0,1,1,1960,None
1,2,1,1965,None
2,3,1,1980,None
3,4,1,1981,None
4,5,1,1982,None
5,6,1,1983,None
6,7,1,1984,None
7,8,1,1985,None
8,9,1,1986,None
9,10,1,1987,None


### 2.1. Перевірка нормалізації

In [12]:
query("""
SELECT
    (SELECT COUNT(*) FROM infectious_cases) AS raw_rows,
    (SELECT COUNT(*) FROM infectious_cases_norm) AS norm_rows,
    (SELECT COUNT(*) FROM entities) AS entities_rows,
    (SELECT COUNT(*) FROM entities WHERE code IS NULL) AS entities_without_code,
    (SELECT SUM(NULLIF("Number_rabies", '')::NUMERIC) FROM infectious_cases) AS raw_rabies_sum,
    (SELECT SUM(number_rabies) FROM infectious_cases_norm) AS norm_rabies_sum
""")

,raw_rows,norm_rows,entities_rows,entities_without_code,raw_rabies_sum,norm_rabies_sum
0,10521,10521,245,34,3.334449e+06,3.334449e+06


In [13]:
query("""
SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name IN ('entities', 'infectious_cases_norm')
ORDER BY table_name, ordinal_position
""")

,table_name,column_name,data_type,is_nullable
0,entities,id,integer,NO
1,entities,entity,text,NO
2,entities,code,text,YES
3,infectious_cases_norm,id,integer,NO
4,infectious_cases_norm,entity_id,integer,NO
5,infectious_cases_norm,year,smallint,NO
6,infectious_cases_norm,number_yaws,bigint,YES
7,infectious_cases_norm,polio_cases,bigint,YES
8,infectious_cases_norm,cases_guinea_worm,bigint,YES
9,infectious_cases_norm,number_rabies,numeric,YES


In [14]:
query("""
SELECT
    e.entity,
    e.code,
    n.year,
    n.polio_cases,
    n.number_rabies
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE e.code = 'UKR'
ORDER BY n.year DESC
LIMIT 5
""")

,entity,code,year,polio_cases,number_rabies
0,Ukraine,UKR,2022,NaN,NaN
1,Ukraine,UKR,2021,NaN,NaN
2,Ukraine,UKR,2020,0.0,NaN
3,Ukraine,UKR,2019,0.0,5.219926
4,Ukraine,UKR,2018,0.0,5.504843


Рядків після нормалізації стільки ж, скільки в сирій таблиці (10 521), і сума Number_rabies однакова. Отже, JOIN нічого не загубив і не продублював. У довіднику 245 записів, з них 34 без коду. Порожні рядки стали `NULL`: в Україні за 2020–2022 роки Number_rabies немає.

## 3. Аналіз `Number_rabies`

In [15]:
query("""
SELECT
    e.id AS entity_id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies,
    COUNT(*) AS years_count
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10
""")

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies,years_count
0,241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11,30
1,120,Lower Middle Income (WB),NaN,15193.96,10202.53,19182.80,455818.78,30
2,202,South Asia (WB),NaN,11729.89,7271.28,15361.88,351896.66,30
3,204,South-East Asia Region (WHO),NaN,11424.33,6806.01,15641.96,342729.81,30
4,77,G20,NaN,10189.05,6339.08,13164.88,305671.38,30
5,97,India,IND,8599.17,5425.87,11121.14,257975.20,30
6,209,Sub-Saharan Africa (WB),NaN,4748.27,4356.82,5017.98,142448.04,30
7,3,African Region (WHO),NaN,4649.70,4283.07,4917.78,139491.07,30
8,121,Low Income (WB),NaN,3568.11,3014.62,3873.53,107043.16,30
9,58,East Asia & Pacific (WB),NaN,3427.46,2009.94,4591.01,102823.83,30


Групування йде за `entity_id`, рядки без значення відкидає `WHERE`. Дані про сказ є лише за 1990–2019 роки, тому в кожного рядка топ-10 рівно 30 років.

Перше місце займає World, у середньому 20 192 випадки на рік. З окремих країн у десятку потрапила тільки Індія (8 599), решта рядків належать групам за доходом, регіонам ВООЗ і Світового банку та G20. Завдання вимагає рахувати для кожної пари Entity/Code, тому агрегати я не прибираю. Для порівняння нижче той самий запит лише для записів із кодом, без World.

In [16]:
query("""
SELECT
    e.id AS entity_id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies,
    COUNT(*) AS years_count
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
  AND e.code IS NOT NULL
  AND e.code <> 'OWID_WRL'
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10
""")

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies,years_count
0,97,India,IND,8599.17,5425.87,11121.14,257975.20,30
1,163,Pakistan,PAK,1582.17,1177.14,1881.73,47465.09,30
2,151,Nigeria,NGA,1335.42,1073.95,1441.12,40062.46,30
3,42,China,CHN,1247.91,744.25,2075.59,37437.20,30
4,70,Ethiopia,ETH,1145.17,758.05,1323.40,34355.18,30
5,143,Myanmar,MMR,972.90,560.70,1449.83,29187.11,30
6,19,Bangladesh,BGD,785.71,180.40,1734.71,23571.26,30
7,146,Nepal,NPL,661.25,338.14,1041.63,19837.53,30
8,170,Philippines,PHL,618.69,358.06,898.07,18560.73,30
9,98,Indonesia,IDN,267.71,124.62,481.37,8031.44,30


Серед країн Індія далеко попереду: 8 599 випадків на рік проти 1 582 у Пакистану, другого в списку, тобто приблизно в 5,4 раза більше. Найсильніше за 30 років змінювались оцінки в Бангладеш: від 180 до 1 735 випадків на рік, майже вдесятеро.

## 4. Колонка різниці в роках

`MAKE_DATE(year, 1, 1)` збирає дату з року, `CURRENT_DATE` дає сьогоднішню дату, а `AGE()` повертає інтервал між ними. З інтервалу `DATE_PART('year', ...)` бере кількість повних років. Дата завжди 1 січня, тому повних років рівно стільки, скільки між номерами років.

Через `CURRENT_DATE` значення в колонках `today` і `years_diff` залежать від дня запуску, тому при кожному запуску ноутбука вони можуть відрізнятися.

In [17]:
query("""
SELECT
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1) AS year_start_date,
    CURRENT_DATE AS today,
    DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INT AS years_diff
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY e.entity, n.year
LIMIT 10
""")

,entity,year,year_start_date,today,years_diff
0,Afghanistan,1960,1960-01-01,2026-10-31,66
1,Afghanistan,1965,1965-01-01,2026-10-31,61
2,Afghanistan,1980,1980-01-01,2026-10-31,46
3,Afghanistan,1981,1981-01-01,2026-10-31,45
4,Afghanistan,1982,1982-01-01,2026-10-31,44
5,Afghanistan,1983,1983-01-01,2026-10-31,43
6,Afghanistan,1984,1984-01-01,2026-10-31,42
7,Afghanistan,1985,1985-01-01,2026-10-31,41
8,Afghanistan,1986,1986-01-01,2026-10-31,40
9,Afghanistan,1987,1987-01-01,2026-10-31,39


## 5. Власні функції

### 5.1. Різниця в роках

Функція повторює вираз з розділу 4. Вона `STABLE`, а не `IMMUTABLE`: `CURRENT_DATE` змінюється від дня до дня, хоча в межах одного запиту стала. Для 1996 року функція повертає 30, і на всіх 10 521 рядках її результат збігається з вбудованим виразом.

In [18]:
execute("""
CREATE OR REPLACE FUNCTION fn_years_since(p_year INT)
RETURNS INT
LANGUAGE sql
STABLE
AS $$
    SELECT DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INT
$$
""")

query("SELECT fn_years_since(1996) AS years_since_1996")

,years_since_1996
0,30


In [19]:
query("""
SELECT
    e.entity,
    n.year,
    DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INT AS years_diff_builtin,
    fn_years_since(n.year) AS years_diff_fn
FROM infectious_cases_norm AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY e.entity, n.year
LIMIT 10
""")

,entity,year,years_diff_builtin,years_diff_fn
0,Afghanistan,1960,66,66
1,Afghanistan,1965,61,61
2,Afghanistan,1980,46,46
3,Afghanistan,1981,45,45
4,Afghanistan,1982,44,44
5,Afghanistan,1983,43,43
6,Afghanistan,1984,42,42
7,Afghanistan,1985,41,41
8,Afghanistan,1986,40,40
9,Afghanistan,1987,39,39


In [20]:
query("""
SELECT COUNT(*) AS mismatches
FROM infectious_cases_norm
WHERE fn_years_since(year) <> DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INT
""")

,mismatches
0,0


### 5.2. Середня кількість захворювань за період

Функція ділить річну кількість захворювань на дільник. Якщо дільник дорівнює 0, `NULLIF` повертає `NULL`, і помилки ділення на нуль не буде. Результат залежить лише від аргументів, тому функція `IMMUTABLE`.

Її викликаємо на сирій таблиці `infectious_cases`, де `Number_rabies` зберігається як текст. Регулярний вираз `'^-?[0-9]+([.][0-9]+)?$'` пропускає до приведення в `NUMERIC` лише числа. Без нього порожній рядок зламав би запит: `''::NUMERIC` дає помилку.

In [21]:
execute("""
CREATE OR REPLACE FUNCTION fn_avg_cases_per_period(p_cases_per_year NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT p_cases_per_year / NULLIF(p_divisor, 0)
$$
""")

query("""
SELECT
    fn_avg_cases_per_period(1200, 12) AS per_month,
    fn_avg_cases_per_period(1200, 4) AS per_quarter,
    fn_avg_cases_per_period(1200, 2) AS per_half_year,
    fn_avg_cases_per_period(1200, 0) AS divisor_zero
""")

,per_month,per_quarter,per_half_year,divisor_zero
0,100.0,300.0,600.0,None


In [22]:
query("""
SELECT
    "Entity",
    "Year",
    "Number_rabies",
    ROUND(fn_avg_cases_per_period("Number_rabies"::NUMERIC, 12), 2) AS rabies_per_month,
    ROUND(fn_avg_cases_per_period("Number_rabies"::NUMERIC, 4), 2) AS rabies_per_quarter,
    ROUND(fn_avg_cases_per_period("Number_rabies"::NUMERIC, 2), 2) AS rabies_per_half_year
FROM infectious_cases
WHERE "Number_rabies" ~ '^-?[0-9]+([.][0-9]+)?$'
ORDER BY "Number_rabies"::NUMERIC DESC
LIMIT 10
""")

,Entity,Year,Number_rabies,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,World,1990,24744.658,2062.05,6186.16,12372.33
1,World,1991,24559.182,2046.60,6139.80,12279.59
2,World,1992,24337.137,2028.09,6084.28,12168.57
3,World,1993,24127.453,2010.62,6031.86,12063.73
4,World,1994,23972.55,1997.71,5993.14,11986.28
5,World,1995,23905.242,1992.10,5976.31,11952.62
6,World,1996,23831.625,1985.97,5957.91,11915.81
7,World,1997,23652.562,1971.05,5913.14,11826.28
8,World,1998,23415.79,1951.32,5853.95,11707.90
9,World,1999,23185.303,1932.11,5796.33,11592.65


In [23]:
query("""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE "Number_rabies" ~ '^-?[0-9]+([.][0-9]+)?$') AS numeric_rows,
    COUNT(*) FILTER (WHERE "Number_rabies" !~ '^-?[0-9]+([.][0-9]+)?$') AS skipped_rows
FROM infectious_cases
""")

,all_rows,numeric_rows,skipped_rows
0,10521,6780,3741


Фільтр пропустив 6 780 числових рядків і відкинув 3 741, як і в профілі колонки з розділу 1.4. Для World у 1990 році 24 744.66 випадку на рік дають 2 062.05 на місяць, 6 186.16 на квартал і 12 372.33 на півріччя.

In [24]:
conn.close()
engine.dispose()